# Avaliação do modelo especialista: Bronquite e Infarto

## Bibliotecas utilizadas

In [2]:
import numpy as np
import os
import re
from matplotlib import pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, fbeta_score

## Definição das classes

In [3]:
actions = ['Bronquite', 'Infarto']
num_frames = 30

#Número que representa cada classe
label_map = {label: num for num, label in enumerate(actions)}

#Nomes das classes nas figuras
labels_plot = ['Bronquite', 'Infarto']

## Parâmetros e pastas

In [4]:
weights_dir = "../models/models/bro_inf"
results_dir = "../results/bro_inf"
cm_dir = os.path.join(results_dir, "matrizes_por_conjunto")
os.makedirs(cm_dir, exist_ok=True)

usar_latex = True #Colocar False se não tiver o Latex instalado

## Carregamento dos dados de validação

In [5]:
#Cada sujeito é um fold da validação cruzada (leave-one-subject-out)
subject_names = ['AL','e0','e1','e2','e3','ME','MV','RO','SA','MI']

num_endings = 20 #Quantidade de conjuntos de teste (as 20 versões sorteadas de cada vídeo)

#Dados de validação de cada conjunto e de cada sujeito
#X: janelas de keypoints dos vídeos
#y: rótulos dos vídeos
subject_data_sets = {
    k: {
        s: {'X': [], 'y': []}
        for s in subject_names
    }
    for k in range(1, num_endings+1)
}

pad = len(str(num_frames - 1)) #Largura do zero à esquerda no nome dos arquivos (00.npy, 01.npy, ...)

#Dados de validação: keypoints sem augmentation, todas as versões de cada vídeo
for action in actions:
    action_dir = os.path.join('../keypoints_data', action)
    for video_folder in os.listdir(action_dir):
        #Identifica o sujeito pelo nome da pasta (ex.: Bronquite_AL_BRO_AL_01_1 -> AL)
        m = re.search(r'_(%s)_' % '|'.join(subject_names), video_folder)
        if not m:
            raise ValueError(f"Não achou sujeito em {video_folder}")
        subj = m.group(1)

        #Identifica o conjunto pelo número no fim do nome (ex.: Bronquite_AL_BRO_AL_01_1 -> conjunto 1)
        m_end = re.search(r'_(\d+)$', video_folder)
        if not m_end:
            continue

        end_num = int(m_end.group(1))
        if end_num < 1 or end_num > num_endings:
            continue

        video_dir = os.path.join(action_dir, video_folder)
        window = []
        for frame_num in range(num_frames):
            fname = f"{frame_num:0{pad}d}.npy"
            fpath = os.path.join(video_dir, fname)
            if not os.path.exists(fpath):
                break
            res = np.load(fpath)
            window.append(res)

        if len(window) == num_frames:
            label = label_map[action]
            subject_data_sets[end_num][subj]['X'].append(window)
            subject_data_sets[end_num][subj]['y'].append(label)

#Converte para array e transforma os rótulos em one-hot
for k in range(1, num_endings+1):
    for s in subject_names:
        X = np.array(subject_data_sets[k][s]['X'])
        y = to_categorical(subject_data_sets[k][s]['y'], num_classes=len(actions))

        subject_data_sets[k][s]['X'] = X
        subject_data_sets[k][s]['y'] = y

## Funções auxiliares

In [6]:
def class_acc(true, pred, labels):
    #Retorna a acurácia de cada classe
    acc = {}
    for i, lbl in enumerate(labels):
        idx = np.where(true == i)[0]
        acc[lbl] = np.mean(pred[idx] == true[idx]) if len(idx) > 0 else 0
    return acc


labels = list(label_map.keys())


def salva_matriz(cm, caminho):
    #Salva em PDF a matriz de confusão normalizada por linha, com a porcentagem e a quantidade de amostras em cada célula
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    #Porcentagens com duas casas decimais e vírgula (ex.: 76,00)
    percent_str = np.vectorize(lambda x: f"{x:0.2f}".replace(".", ","))(np.round(cm_norm * 100, 2))

    #Símbolo de porcentagem (com LaTeX é preciso usar a barra)
    simbolo = "\\%" if usar_latex else "%"

    #Texto de cada célula: porcentagem e quantidade de amostras (ex.: 76,00% e 38)
    annot_text = np.empty_like(percent_str, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot_text[i, j] = f"{percent_str[i, j]}{simbolo}\n{cm[i, j]}"

    #Fontes LaTeX nas figuras (requer LaTeX instalado)
    with plt.rc_context({"text.usetex": usar_latex, "font.family": "serif"}):
        plt.figure(figsize=(5,5))

        ax = sns.heatmap(
            cm_norm,
            annot=annot_text,
            fmt='',
            cmap='Blues',
            xticklabels=labels_plot,
            yticklabels=labels_plot,
            cbar=False,
            annot_kws={"size": 20},
            square=True
        )

        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=20)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

        plt.tight_layout()
        plt.savefig(caminho)
        plt.close()

## Avaliação dos 20 conjuntos de teste

In [7]:
set_means = []
per_set_class_accs = []
geral_class_accs = []

#Métricas por conjunto
per_set_precision = []
per_set_recall = []
per_set_f1 = []
per_set_f2 = []

macro_precision_sets = []
macro_recall_sets = []
macro_f1_sets = []
macro_f2_sets = []


for k in range(1, num_endings + 1): #Avalia cada conjunto de teste com os modelos dos 10 folds
    print(f"\n========== Conjunto {k} ==========")

    yhat_all = []
    ytrue_all = []
    class_fold_results = []

    for i, val_subj in enumerate(subject_names):

        X_val = subject_data_sets[k][val_subj]['X']
        y_val = subject_data_sets[k][val_subj]['y']

        if len(X_val) == 0:
            print(f"Atenção: {val_subj} não possui amostras no conjunto {k}.")
            continue

        #O modelo do fold i foi treinado sem o sujeito i
        model = load_model(f'{weights_dir}/model_fold_{i+1}.keras')

        y_pred = np.argmax(model.predict(X_val), axis=1)
        y_true = np.argmax(y_val, axis=1)

        yhat_all.append(y_pred)
        ytrue_all.append(y_true)

        acc_fold = np.mean(y_pred == y_true)
        print(f"Fold {i+1} – {val_subj}: acc = {acc_fold:.4f}")

        cls_acc = class_acc(y_true, y_pred, labels)
        class_fold_results.append(cls_acc)

        for lbl, a in cls_acc.items():
            print(f"  {lbl}: {a:.4f}")

    if len(yhat_all) == 0:
        print(f"\nConjunto {k} vazio — sem métricas.\n")
        continue

    y_true_final = np.concatenate(ytrue_all)
    y_pred_final = np.concatenate(yhat_all)

    #Acurácia geral do conjunto
    acc_global_k = np.mean(y_pred_final == y_true_final)
    set_means.append(acc_global_k)

    print(f"\nAcurácia geral do conjunto {k}: {acc_global_k:.4f}")

    #Acurácia por classe (média entre os folds)
    per_class_lists = {lbl: [] for lbl in labels}

    for fold_dict in class_fold_results:
        for lbl in labels:
            per_class_lists[lbl].append(fold_dict[lbl])

    class_mean_k = {lbl: np.mean(per_class_lists[lbl]) for lbl in labels}
    class_std_k = {lbl: np.std(per_class_lists[lbl]) for lbl in labels}

    per_set_class_accs.append(class_mean_k)

    #Precisão, recall, F1 e F2 por classe
    precisions_k = precision_score(y_true_final, y_pred_final, average=None, zero_division=0)
    recalls_k = recall_score(y_true_final, y_pred_final, average=None, zero_division=0)
    f1s_k = f1_score(y_true_final, y_pred_final, average=None, zero_division=0)
    f2s_k = fbeta_score(y_true_final, y_pred_final, beta=2, average=None, zero_division=0)

    per_set_precision.append(dict(zip(labels, precisions_k)))
    per_set_recall.append(dict(zip(labels, recalls_k)))
    per_set_f1.append(dict(zip(labels, f1s_k)))
    per_set_f2.append(dict(zip(labels, f2s_k)))

    print("\nMétricas por classe – Conjunto", k)
    for i, lbl in enumerate(labels):
        print(f"  {lbl}: P={precisions_k[i]:.4f} | R={recalls_k[i]:.4f} | F1={f1s_k[i]:.4f} | F2={f2s_k[i]:.4f}")

    #Métricas macro (média entre as classes)
    macro_precision_sets.append(
        precision_score(y_true_final, y_pred_final, average='macro', zero_division=0)
    )
    macro_recall_sets.append(
        recall_score(y_true_final, y_pred_final, average='macro', zero_division=0)
    )
    macro_f1_sets.append(
        f1_score(y_true_final, y_pred_final, average='macro', zero_division=0)
    )
    macro_f2_sets.append(
        fbeta_score(y_true_final, y_pred_final, beta=2, average='macro', zero_division=0)
    )

    print("\nMétricas macro – Conjunto", k)
    print(f"  Precisão: {macro_precision_sets[-1]:.4f}")
    print(f"  Recall:   {macro_recall_sets[-1]:.4f}")
    print(f"  F1:       {macro_f1_sets[-1]:.4f}")
    print(f"  F2:       {macro_f2_sets[-1]:.4f}")

    #Matriz de confusão do conjunto, salva em results_dir/matrizes_por_conjunto
    cm = confusion_matrix(y_true_final, y_pred_final)
    salva_matriz(cm, os.path.join(cm_dir, f'conjunto_{k}.pdf'))


#Resultados finais entre os conjuntos
print("\n================ RESULTADO FINAL ================")
print(f"Acurácia geral média: {np.mean(set_means):.4f}")
print(f"Desvio padrão:        {np.std(set_means):.4f}")

def mean_std(metric_sets):
    return np.mean(metric_sets), np.std(metric_sets)

print("\n=========== MÉTRICAS MACRO FINAIS ===========")
print(f"Precisão: média={*mean_std(macro_precision_sets),}")
print(f"Recall:   média={*mean_std(macro_recall_sets),}")
print(f"F1:       média={*mean_std(macro_f1_sets),}")
print(f"F2:       média={*mean_std(macro_f2_sets),}")

print("\n=========== MÉTRICAS POR CLASSE (MÉDIA ± STD) ===========")
for lbl in labels:
    p = [d[lbl] for d in per_set_precision]
    r = [d[lbl] for d in per_set_recall]
    f1 = [d[lbl] for d in per_set_f1]
    f2 = [d[lbl] for d in per_set_f2]

    print(lbl)
    print(f"  Precisão: {np.mean(p):.4f} ± {np.std(p):.4f}")
    print(f"  Recall:   {np.mean(r):.4f} ± {np.std(r):.4f}")
    print(f"  F1:       {np.mean(f1):.4f} ± {np.std(f1):.4f}")
    print(f"  F2:       {np.mean(f2):.4f} ± {np.std(f2):.4f}")


========== Conjunto 1 ==========


E0000 00:00:1791808530.098612   10505 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1791808530.107086   10505 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step
Fold 1 – AL: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
Fold 2 – e0: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
Fold 3 – e1: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
Fold 4 – e2: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
Fold 5 – e3: acc = 0.9000
  Bronquite: 1.0000
  Infarto: 0.8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step
Fold 6 – ME: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step
Fold 7 – MV: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
Fold 8 – RO: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step
Fold 9 – SA: acc = 0.7000
  Bronquite: 1.0000
  Infarto: 0.4000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step
Fold 10 – MI: acc = 1.0000
  Bronq

### Acurácia por classe entre os conjuntos

In [8]:
#Média e desvio por classe entre os conjuntos
#folds: média da acurácia de cada fold (per_set_class_accs)
#agrupado: acurácia com as predições dos 10 folds juntas, igual ao recall (per_set_recall)
print(f"{'Classe':20s} | média folds | média agrupada | desvio folds | desvio agrupado")
for lbl in labels:
    folds = [d[lbl] * 100 for d in per_set_class_accs]
    agrupado = [d[lbl] * 100 for d in per_set_recall]
    print(f"{lbl:20s} | {np.mean(folds):11.2f} | {np.mean(agrupado):14.2f} | {np.std(folds):12.2f} | {np.std(agrupado):15.2f}")

Classe               | média folds | média agrupada | desvio folds | desvio agrupado
Bronquite            |       99.00 |          98.96 |         1.48 |            1.55
Infarto              |       91.30 |          91.30 |         2.78 |            2.78


### Ranking dos conjuntos

In [9]:
#Ranking dos conjuntos pela acurácia geral
ranking = sorted(enumerate(set_means, start=1), key=lambda x: x[1], reverse=True)
print("Conjunto | Acurácia geral")
for k, acc in ranking:
    print(f"{k:8d} | {acc:.4f}")

Conjunto | Acurácia geral
      16 | 0.9796
       4 | 0.9694
       1 | 0.9592
       2 | 0.9592
       3 | 0.9592
       5 | 0.9592
       8 | 0.9592
       9 | 0.9592
      14 | 0.9592
      17 | 0.9592
      20 | 0.9592
      10 | 0.9490
      11 | 0.9490
      12 | 0.9490
       6 | 0.9388
       7 | 0.9388
      13 | 0.9388
      15 | 0.9388
      19 | 0.9184
      18 | 0.9082


## Conjunto escolhido (melhor conjunto)

### Escolha do conjunto

In [10]:
k_escolhido = 16

### Predições dos 10 folds

In [11]:
print(f"\n========== Avaliando SOMENTE o Conjunto {k_escolhido} ==========\n")

yhat_all = []
ytrue_all = []
class_fold_results = []
fold_accs = [] #Acurácia de cada fold

for i, val_subj in enumerate(subject_names):

    X_val = subject_data_sets[k_escolhido][val_subj]['X']
    y_val = subject_data_sets[k_escolhido][val_subj]['y']

    if len(X_val) == 0:
        print(f"Atenção: {val_subj} não possui amostras no conjunto {k_escolhido}.")
        continue

    model = load_model(f'{weights_dir}/model_fold_{i+1}.keras')

    y_pred = np.argmax(model.predict(X_val), axis=1)
    y_true = np.argmax(y_val, axis=1)

    yhat_all.append(y_pred)
    ytrue_all.append(y_true)

    acc_fold = np.mean(y_pred == y_true)
    fold_accs.append(acc_fold)
    print(f"Fold {i+1} – {val_subj}: acc = {acc_fold:.4f}")

    cls_acc = class_acc(y_true, y_pred, labels)
    class_fold_results.append(cls_acc)

    for lbl, a in cls_acc.items():
        print(f"  {lbl}: {a:.4f}")


if len(yhat_all) == 0:
    raise ValueError(f"O conjunto {k_escolhido} está vazio")

#Junta as predições dos 10 folds
y_true_final = np.concatenate(ytrue_all)
y_pred_final = np.concatenate(yhat_all)

#Acurácia geral do conjunto
acc_global = np.mean(y_pred_final == y_true_final)
print(f"\nAcurácia GLOBAL do Conjunto {k_escolhido}: {acc_global:.4f}\n")

#Média e desvio padrão da acurácia entre os folds, em porcentagem
print(f"Acurácia média entre os folds: {np.mean(fold_accs) * 100:.2f}")
print(f"Desvio padrão entre os folds:  {np.std(fold_accs) * 100:.2f}\n")

#Acurácia por classe (média e desvio entre os folds)
per_class_lists = {lbl: [] for lbl in labels}

for fold_dict in class_fold_results:
    for lbl in labels:
        per_class_lists[lbl].append(fold_dict[lbl])

class_mean = {lbl: np.mean(per_class_lists[lbl]) for lbl in labels}
class_std = {lbl: np.std(per_class_lists[lbl]) for lbl in labels}

print("\nAcurácias por classe:")
for lbl in labels:
    print(f"  {lbl}: média = {class_mean[lbl]:.4f} | desvio = {class_std[lbl]:.4f}")

#Matriz de confusão (contagens) e normalizada por linha (porcentagem de cada classe real)
cm = confusion_matrix(y_true_final, y_pred_final)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
percent_vals = np.round(cm_norm * 100, 2)


========== Avaliando SOMENTE o Conjunto 16 ==========

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step
Fold 1 – AL: acc = 0.9000
  Bronquite: 0.8000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
Fold 2 – e0: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
Fold 3 – e1: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step
Fold 4 – e2: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step
Fold 5 – e3: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step
Fold 6 – ME: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 599ms/step
Fold 7 – MV: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step
Fold 8 – RO: acc = 1.0000
  Bronquite: 1.0000
  Infarto: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step
Fold 9 – SA: acc = 0.9000
  Bronquite: 1.0000
  Infarto: 0.8000
1/1 ━━━━━━━━━━━━━

### Métricas por classe

In [12]:
print("\n=========== MÉTRICAS FINAIS POR CLASSE ===========\n")

precisions = precision_score(y_true_final, y_pred_final, average=None, zero_division=0)
recalls = recall_score(y_true_final, y_pred_final, average=None, zero_division=0)
f1s = f1_score(y_true_final, y_pred_final, average=None, zero_division=0)
f2s = fbeta_score(y_true_final, y_pred_final, beta=2, average=None, zero_division=0)

for i, lbl in enumerate(labels):
    print(f"Classe: {lbl}")
    print(f"  Precisão:  {precisions[i]:.4f}")
    print(f"  Recall:    {recalls[i]:.4f}")
    print(f"  F1-score:  {f1s[i]:.4f}")
    print(f"  F2-score:  {f2s[i]:.4f}")
    print()


=========== MÉTRICAS FINAIS POR CLASSE ===========

Classe: Bronquite
  Precisão:  0.9792
  Recall:    0.9792
  F1-score:  0.9792
  F2-score:  0.9792

Classe: Infarto
  Precisão:  0.9800
  Recall:    0.9800
  F1-score:  0.9800
  F2-score:  0.9800



### Métricas macro

In [13]:
precision_macro = precision_score(y_true_final, y_pred_final, average='macro', zero_division=0)
recall_macro = recall_score(y_true_final, y_pred_final, average='macro', zero_division=0)
f1_macro = f1_score(y_true_final, y_pred_final, average='macro', zero_division=0)
f2_macro = fbeta_score(y_true_final, y_pred_final, beta=2, average='macro', zero_division=0)

print(f"  Precisão:  {precision_macro:.4f}")
print(f"  Recall:    {recall_macro:.4f}")
print(f"  F1-score:  {f1_macro:.4f}")
print(f"  F2-score:  {f2_macro:.4f}")
print()

  Precisão:  0.9796
  Recall:    0.9796
  F1-score:  0.9796
  F2-score:  0.9796

